# Assignment 1 — South German Credit
Data Science for Business · 2026–2027

Student 1: name and r-number
Student 2: name and r-number

## 0. Load the dataset

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, auc
from IPython.display import display

In [ ]:
DATA_PATH = Path("South_German_Credit.csv")
if not DATA_PATH.is_file():
    raise FileNotFoundError("Upload South_German_Credit.csv to the Colab Files panel.")
credit = pd.read_csv(DATA_PATH)
display(credit.head())

## 1. Explore the data

In [ ]:
# 1. Check data
print(f"Dataset shape: {credit.shape}")
print(f"Missing values:\n{credit.isnull().sum().sum()}")

# 2. Target Creation (1 = bad, 0 = good)
credit['target'] = credit['credit_risk'].apply(lambda x: 1 if x == 0 else 0)

# 3. Train/Test Split (70/30 Stratified)
SEED = 2026
train, test = train_test_split(
    credit, test_size=0.30, random_state=SEED, stratify=credit['target']
)
print(f"Train size: {len(train)}, Test size: {len(test)}")

In [ ]:
# 4. Joint Groups for two variables
train = train.copy()
train['duration_bin'] = pd.cut(train['duration'], bins=[0, 12, 24, 36, 100], labels=['<=12m', '13-24m', '25-36m', '>36m'])
group_stats = train.groupby(['credit_history', 'duration_bin'])['target'].agg(bad_rate='mean', n='size').reset_index().dropna()

fig, ax1 = plt.subplots(figsize=(10, 6))
ax2 = ax1.twinx()
x_labels = group_stats['credit_history'].astype(str) + " & " + group_stats['duration_bin'].astype(str)
x_pos = np.arange(len(x_labels))
bars = ax1.bar(x_pos, group_stats['n'], color='lightblue', label='Group Size (n)')
lines = ax2.plot(x_pos, group_stats['bad_rate'], color='red', marker='o', linewidth=2, label='Bad Credit Rate')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(x_labels, rotation=45, ha='right')
ax1.set_xlabel('Credit History & Duration Bin')
ax1.set_ylabel('Number of Observations (n)')
ax2.set_ylabel('Bad Credit Rate')
ax1.set_title('Bad Credit Rate and Group Size by Credit History & Duration')
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper left')
plt.tight_layout()
plt.show()

## 2. Build and evaluate the model

In [ ]:
# Predictor Selection, WOE, and IV
def calculate_woe_iv(df, feature, target):
    lst = []
    for val in df[feature].unique():
        val_df = df[df[feature] == val]
        good = len(val_df[val_df[target] == 0])
        bad = len(val_df[val_df[target] == 1])
        lst.append({'Value': val, 'Good': good, 'Bad': bad})
    dset = pd.DataFrame(lst)
    dset['Distr_Good'] = dset['Good'] / dset['Good'].sum()
    dset['Distr_Bad'] = dset['Bad'] / dset['Bad'].sum()
    dset['WOE'] = np.log(dset['Distr_Good'] / dset['Distr_Bad'])
    dset['IV'] = (dset['Distr_Good'] - dset['Distr_Bad']) * dset['WOE']
    dset = dset.replace([np.inf, -np.inf], 0)
    return dict(zip(dset['Value'], dset['WOE'])), dset['IV'].sum()

train['amount_bin'] = pd.qcut(train['amount'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
features = ['duration_bin', 'amount_bin', 'credit_history', 'savings']
woe_dicts, iv_values = {}, {}

for feat in features:
    woe_dict, iv = calculate_woe_iv(train, feat, 'target')
    woe_dicts[feat] = woe_dict
    iv_values[feat] = iv
    train[f'{feat}_WOE'] = train[feat].map(woe_dict)

print("Information Values (IV):", iv_values)

In [ ]:
# Fitted Model and ROC Evaluation
woe_features = [f'{feat}_WOE' for feat in features]
X_train = train[woe_features].fillna(0)
y_train = train['target']

model = LogisticRegression(random_state=SEED)
model.fit(X_train, y_train)

# Test Set Preparation
test = test.copy()
test['duration_bin'] = pd.cut(test['duration'], bins=[0, 12, 24, 36, 100], labels=['<=12m', '13-24m', '25-36m', '>36m'])
test['amount_bin'] = pd.qcut(test['amount'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
for feat in features:
    test[f'{feat}_WOE'] = test[feat].map(woe_dicts[feat]).fillna(0)

X_test = test[woe_features]
y_test = test['target']

fpr_train, tpr_train, _ = roc_curve(y_train, model.predict_proba(X_train)[:, 1])
fpr_test, tpr_test, _ = roc_curve(y_test, model.predict_proba(X_test)[:, 1])
plt.figure(figsize=(8, 6))
plt.plot(fpr_train, tpr_train, label=f'Train AUC = {auc(fpr_train, tpr_train):.3f}')
plt.plot(fpr_test, tpr_test, label=f'Test AUC = {auc(fpr_test, tpr_test):.3f}')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.show()

## 3. Construct the scorecard

In [ ]:
# Scorecard Construction
Target_Score = 600
Target_Odds = 50 
PDO = 20
Factor = PDO / np.log(2)
Offset = Target_Score - Factor * np.log(Target_Odds)
n_features = len(features)

scorecard = []
for i, feat in enumerate(features):
    woe_dict = woe_dicts[feat]
    coef = model.coef_[0][i]
    for level, woe in woe_dict.items():
        score = (Offset / n_features) + (woe * coef * Factor) + ((model.intercept_[0] * Factor) / n_features)
        scorecard.append({'Predictor': feat, 'Level': level, 'WOE': woe, 'Points': round(score)})

scorecard_df = pd.DataFrame(scorecard)
display(scorecard_df)

In [ ]:
# Boxplots and Score Verification
def calculate_score(row):
    total = 0
    for feat in features:
        val = row[feat]
        point = scorecard_df[(scorecard_df['Predictor'] == feat) & (scorecard_df['Level'] == val)]['Points']
        if not point.empty:
            total += point.values[0]
    return total

test['Score'] = test.apply(calculate_score, axis=1)

plt.figure(figsize=(8, 6))
good_scores = test[test['target'] == 0]['Score']
bad_scores = test[test['target'] == 1]['Score']
plt.boxplot([good_scores, bad_scores], labels=['Good Credit (0)', 'Bad Credit (1)'])
highest_bad, lowest_good = bad_scores.max(), good_scores.min()
plt.scatter([2], [highest_bad], color='red', zorder=10, label=f'Highest Bad ({highest_bad})')
plt.scatter([1], [lowest_good], color='blue', zorder=10, label=f'Lowest Good ({lowest_good})')
plt.ylabel('Credit Score')
plt.title('Distribution of Test Scores by Credit Outcome')
plt.legend()
plt.show()